In [ ]:
import os

import pandas
import xarray
from tqdm import tqdm

In [ ]:
figure_directory = "./figures"
os.makedirs(figure_directory, exist_ok=True)

### Data Ingestion

Copy all the relevant files from the /ETL/data folder after running the ETL pipeline.

#### Annual Electricity Demand

In [ ]:
electricity_annual_demand_folder = "./data/annual_electricity_demand/"
ed_annual_files = [
    file_name
    for file_name in os.listdir(electricity_annual_demand_folder)
    if file_name.endswith(".parquet")
]
ed_annual_files[:5]
df_annual_demand = pandas.DataFrame()

for file_name in tqdm(ed_annual_files):
    df_current = pandas.read_parquet(
        electricity_annual_demand_folder + file_name
    )

    df_current = df_current.resample(
        "1h", label="right", closed="right"
    ).mean()

    # Add a column for the region name
    df_current["region_code"] = file_name.split(".")[0]

    # Reset index to move "Time (UTC)" to a column
    df_current = df_current.reset_index()

    df_annual_demand = pandas.concat(
        [df_annual_demand, df_current], ignore_index=True
    )
print(df_annual_demand.shape)
df_annual_demand.head()

#### Temperature Data

In [ ]:
temperature_folder = "./data/temperature/"
temperature_files = [
    file_name
    for file_name in os.listdir(temperature_folder)
    if file_name.endswith(".parquet")
]
temperature_files[:5]
df_all_temperature = pandas.DataFrame()

for file_name in tqdm(temperature_files):
    df_current = pandas.read_parquet(temperature_folder + file_name)

    # Add a column for the region name
    df_current["region_code"] = file_name.split("_temp")[0]

    # Reset index to move "Time (UTC)" to a column
    df_current = df_current.reset_index()

    df_all_temperature = pandas.concat(
        [df_all_temperature, df_current], ignore_index=True
    )
print(df_all_temperature.shape)
df_all_temperature.head()

#### GDP Data

In [ ]:
gdp_folder = "./data/gdp/"
gdp_files = [
    file_name
    for file_name in os.listdir(gdp_folder)
    if file_name.endswith(".nc")
]
df_gdp_data = pandas.DataFrame()

for file_name in tqdm(gdp_files):
    # Extract region code from filename
    # Assuming a format like "US_0.25_deg_2020.nc"
    region_code = file_name.split("_0.25_deg_")[0]
    year = int(file_name.split("_0.25_deg_")[-1].replace(".nc", ""))

    # Open the NetCDF file
    gdp_data = xarray.open_dataset(gdp_folder + file_name)

    # Extract GDP value - assuming the GDP is a variable called 'gdp'
    gdp_value = float(gdp_data.gdp.to_numpy().sum())

    # Create a DataFrame for this file
    df_current = pandas.DataFrame(
        {"year": [year], "GDP": [gdp_value], "region_code": [region_code]}
    )

    # Extract country code (assuming it's the first part of region_code)
    country_code = region_code.split("_")[0]
    df_current["country_code"] = country_code

    # Append to the main DataFrame
    df_gdp_data = pandas.concat([df_gdp_data, df_current], ignore_index=True)

print(df_gdp_data.shape)
df_gdp_data.head()

#### Electricity Demand Data

In [ ]:
electricity_demand_folder = "./data/electricity_demand/"
demand_files = [
    file_name
    for file_name in os.listdir(electricity_demand_folder)
    if file_name.endswith(".parquet")
]
df_demand = pandas.DataFrame()

for file_name in tqdm(demand_files):
    df_current = pandas.read_parquet(electricity_demand_folder + file_name)

    df_current["Load (MW)"] = df_current["Load (MW)"].astype(float)

    df_current = df_current.resample(
        "1h", label="right", closed="right"
    ).mean()

    # Add a column for the region name
    df_current["region_code"] = str.join("_", file_name.split("_")[:-1])

    # Reset index to move "Time (UTC)" to a column
    df_current = df_current.reset_index()

    df_demand = pandas.concat([df_demand, df_current], ignore_index=True)
print(df_demand.shape)
df_demand.head()

#### Combine all datasets

In [ ]:
df_annual_demand = df_annual_demand.sort_values(by=["Time (UTC)"])
df_all_temperature = df_all_temperature.sort_values(by=["Time (UTC)"])
df_demand = df_demand.sort_values(by=["Time (UTC)"])

In [ ]:
# Merge the demand data
combined_dataset = pandas.merge(
    df_all_temperature, df_demand, on=["Time (UTC)", "region_code"]
)
print(combined_dataset.shape)
combined_dataset.head()

In [ ]:
# Run to add annual demand data to the dataset
combined_dataset = pandas.merge(
    combined_dataset, df_annual_demand, on=["Time (UTC)", "region_code"]
)
# Scale the yearly demand from TW to MW
combined_dataset["year_electricity_demand_mw"] = (
    combined_dataset["Annual electricity demand (TWh)"] * 1000000
)
combined_dataset = combined_dataset.drop(
    columns=["Annual electricity demand (TWh)"]
)
print(combined_dataset.shape)
combined_dataset.head()

In [ ]:
# Run to add GDP data to the dataset
combined_dataset = pandas.merge(
    combined_dataset,
    df_gdp_data.drop(columns=["country_code"]),
    left_on=["Local year", "region_code"],
    right_on=["year", "region_code"],
)
combined_dataset = combined_dataset.drop(columns=["year"])
print(combined_dataset.shape)
combined_dataset.head()

In [ ]:
# Remove duplicates
row_count = len(combined_dataset)
print("Before removing duplicates:", row_count)
combined_dataset = combined_dataset.drop_duplicates(
    subset=[col for col in combined_dataset.columns if col != "Load (MW)"]
)
print("Without duplicates: ", len(combined_dataset))
print("Difference", row_count - len(combined_dataset))

In [ ]:
# Remove NaN values
row_count = len(combined_dataset)
print("Before removing NaN values:", row_count)
combined_dataset = combined_dataset.dropna()
print("Without duplicates: ", len(combined_dataset))
print("Difference", row_count - len(combined_dataset))

In [ ]:
combined_dataset = combined_dataset.rename(
    columns={
        "Time (UTC)": "time_utc",
        "Local hour of the day": "local_hour",
        "Local weekend indicator": "is_weekend",
        "Local month of the year": "local_month",
        "Local year": "local_year",
        "Temperature - Top 1 (K)": "year_temp_top1",
        "Temperature - Top 3 (K)": "year_temp_top3",
        "Monthly average temperature - Top 1 (K)": "monthly_temp_avg_top1",
        "Monthly average temperature rank - Top 1": "monthly_temp_avg_rank_top1",
        "Annual average temperature - Top 1 (K)": "year_temp_avg_top1",
        "5 percentile temperature - Top 1 (K)": "year_temp_percentile_5",
        "95 percentile temperature - Top 1 (K)": "year_temp_percentile_95",
        "Annual electricity demand (TWh)": "year_electricity_demand",
        "Annual electricity demand per capita (MWh)": "year_electricity_demand_per_capita_mwh",
        "Load (MW)": "load_mw",
        "GDP": "year_gdp",
    }
)
print(combined_dataset.shape)
combined_dataset.head()

In [ ]:
# Investigate the distribution of available hours per region and year
list_amount_hours_region = []
for name, group in combined_dataset.groupby(["region_code", "local_year"]):
    list_amount_hours_region.append([name[0], name[1], len(group)])

df_amount_hours_region = pandas.DataFrame(
    list_amount_hours_region,
    columns=["region_code", "local_year", "count_available_hours"],
)

df_amount_hours_region["count_available_hours"].hist(bins=10)

In [ ]:
# Overview of the dataset
combined_dataset.head()

In [ ]:
# Introduce a new column that takes load_mw per year and
# calculates for each hour the percentage of the load_mw for that hour

for name, group in combined_dataset.groupby(["region_code", "local_year"]):
    yearly_load = group["load_mw"].sum()
    amount_of_hours_tracked = len(group["load_mw"])
    # Calculate the amount of hours in the specified year
    # accounting for leap years
    amount_of_hours_in_year = (
        len(
            pandas.date_range(start=f"{name[1]}-01-01", end=f"{name[1]}-12-31")
        )
        * 24
    )

    # Calculate the percentage that load_mw represents of yearly load
    load_mw_percentage = group["load_mw"] / yearly_load

    # Adjust the percentages to account for missing hours
    combined_dataset.loc[group.index, "load_mw_percentage"] = (
        load_mw_percentage
        * (amount_of_hours_tracked / amount_of_hours_in_year)
    )

In [ ]:
combined_dataset.to_parquet(
    "./data/processed_dataset_only_temp.parquet", engine="pyarrow"
)
del combined_dataset

### Split into train, test, and validation datasets

In [ ]:
# Read in the dataset
processed_dataset = pandas.read_parquet(
    "./data/processed_dataset_only_temp.parquet", engine="pyarrow"
)

In [ ]:
# Initialize empty dataframes for test and validation sets
test_set = pandas.DataFrame()
test_set_indices = []
validation_set = pandas.DataFrame()
validation_set_indices = []

for name, group in processed_dataset.groupby("region_code"):
    # Keep track of the last available year for each region
    max_year = group["local_year"].max()

    # Select the test set by selecting the last year
    group_test_set = group[group["local_year"] == max_year].copy()
    test_set_indices.append(group_test_set.index)
    test_set = pandas.concat([test_set, group_test_set], ignore_index=True)

    # Select the validation set by selecting the second last year
    group_val_set = group[group["local_year"] == max_year - 1].copy()
    validation_set_indices.append(group_val_set.index)
    validation_set = pandas.concat(
        [validation_set, group_val_set], ignore_index=True
    )

print(
    "Test set size:",
    round((len(test_set) / len(processed_dataset)) * 100, 2),
    "% of total dataset",
)
print(
    "Validation set size:",
    round((len(validation_set) / len(processed_dataset)) * 100, 2),
    "% of total dataset",
)

In [ ]:
# Obtain the indicies of the test and validation sets
all_test_set_indices = [
    index for list_indicies in test_set_indices for index in list_indicies
]
all_val_set_indices = [
    index
    for list_indicies in validation_set_indices
    for index in list_indicies
]

In [ ]:
# Drop test and validation sets from the combined dataset
print("Size before:", len(processed_dataset))
train_set = processed_dataset.drop(index=all_test_set_indices).copy()
train_set = train_set.drop(index=all_val_set_indices)
print("After removing test and val sets:", len(train_set))

In [ ]:
train_set.head()

In [ ]:
def prepare_data(dataset: pandas.DataFrame):
    """
    Process the dataset into splits to be used in training the model.

    Returns
    -------
    features : pandas.DataFrame
        Features for the model.
    target : pandas.Series
        Column with the target variable.
    groups : pandas.Series
        Column containing the region codes
    """
    features = dataset[
        [
            "local_hour",
            "is_weekend",
            "local_month",
            "year_temp_top1",
            "year_temp_top3",
            "monthly_temp_avg_top1",
            "monthly_temp_avg_rank_top1",
            "year_temp_avg_top1",
            "year_temp_percentile_5",
            "year_temp_percentile_95",
            # "year_electricity_demand_per_capita_mwh",
            # "year_gdp",
        ]
    ].copy()

    categorical_features = [
        "local_hour",
        "is_weekend",
        "local_month",
        "monthly_temp_avg_rank_top1",
    ]

    for cat_feature in categorical_features:
        features[cat_feature] = features[cat_feature].astype("category")

    target = dataset["load_mw_percentage"].copy()
    groups = dataset["region_code"].copy()

    return features, target, groups

In [ ]:
train_features, train_target, train_groups = prepare_data(train_set)
val_features, val_target, val_groups = prepare_data(validation_set)

test_features, test_target, test_groups = prepare_data(test_set)

In [ ]:
train_features.head()

In [ ]:
train_target

### Training

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error
from xgboost import XGBRegressor

In [ ]:
# Initialize the XGBoost regressor
xgb_model = XGBRegressor(
    random_state=42,
    enable_categorical=True,
    eval_metric=mean_absolute_percentage_error,
)

In [ ]:
# Train the model
xgb_model.fit(
    train_features, train_target, eval_set=[(val_features, val_target)]
)

In [ ]:
xgb_model.save_model("./data/xgboost_model_only_temp.bin")

#### Prediction on test set

In [ ]:
def calculate_test_error_metric(
    error_metric,
    error_metric_name: str,
    test_set,
    test_predictions,
    test_target,
    message: str = "",
) -> pandas.DataFrame:
    """
    Calcuate the mean absolute percentage error for the test set.

    Returns
    -------
    pandas.DataFrame
        A DataFrame with the region codes, years,
        and mean absolute percentage errors for the test set.
    """
    list_test_metric_values = []
    for name, group in test_set.groupby(["local_year", "region_code"]):
        current_metric = error_metric(
            test_predictions[group.index], test_target.iloc[group.index]
        )

        list_test_metric_values.append([name[1], name[0], current_metric])

    df_validation_metric_values = pandas.DataFrame(
        list_test_metric_values,
        columns=["region_code", "year", error_metric_name],
    )

    df_validation_metric_values.to_parquet(
        "data/test_" + error_metric_name + "_values" + message + ".parquet",
        engine="pyarrow",
    )
    df_validation_metric_values.to_csv(
        "data/test_" + error_metric_name + "_values" + message + ".csv"
    )

    return df_validation_metric_values

In [ ]:
# Predict on test set and calculate MAPE
test_predictions = xgb_model.predict(test_features)
calculate_test_error_metric(
    mean_absolute_percentage_error,
    "MAPE",
    test_set,
    test_predictions,
    test_target,
    "_only_temp",
)

In [ ]:
train_predictions = xgb_model.predict(train_features)

In [ ]:
calculate_test_error_metric(
    mean_absolute_percentage_error,
    "MAPE",
    train_set.reset_index(),
    train_predictions,
    train_target,
    "_train_only_temp",
)

#### Cross validate

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, cross_validate

In [ ]:
cv_features, cv_target, cv_groups = prepare_data(processed_dataset)

In [ ]:
# Perform cross-validation and store results
cv_results = cross_validate(
    xgb_model,
    cv_features,
    cv_target,
    groups=cv_groups,
    cv=LeaveOneGroupOut(),
    scoring=["neg_mean_absolute_percentage_error"],
    return_train_score=True,
    return_indices=True,
    return_estimator=True,
    n_jobs=1,
)

In [ ]:
cv_results.keys()

In [ ]:
cv_results["indices_train"] = cv_results["indices"]["train"]
cv_results["indices_test"] = cv_results["indices"]["test"]

In [ ]:
# Assuming 'cv_results' is your original dictionary and
# 'indices' is the key you want to exclude
cv_results_filtered = {k: v for k, v in cv_results.items() if k != "indices"}

# Create DataFrame from the filtered dictionary
df_cv_results = pandas.DataFrame(cv_results_filtered)

df_cv_results["test_MAPE"] = -df_cv_results[
    "test_neg_mean_absolute_percentage_error"
]
df_cv_results["train_MAPE"] = -df_cv_results[
    "train_neg_mean_absolute_percentage_error"
]

In [ ]:
df_cv_results.head()

In [ ]:
list_test_group_id = []
for test_indices in cv_results["indices"]["test"]:
    list_test_group_id.append(cv_groups.iloc[test_indices[0]])

df_cv_results["group_id"] = list_test_group_id

In [ ]:
df_cv_results.head()

In [ ]:
df_cv_output = df_cv_results[
    ["group_id", "train_MAPE", "test_MAPE", "fit_time", "score_time"]
]

In [ ]:
df_cv_output.to_parquet(
    "./data/cv_results_only_temp.parquet", engine="pyarrow"
)
df_cv_output.to_csv("./data/cv_results_only_temp.csv")

### Synthetic data for all collected data

In [ ]:
entire_dataset = pandas.read_parquet(
    "./data/combined_dataset.parquet", engine="pyarrow"
)

In [ ]:
trained_xgb_model = XGBRegressor()
trained_xgb_model.load_model("./data/xgboost_model.bin")

In [ ]:
entire_dataset.head()

In [ ]:
input_features_columns = trained_xgb_model.feature_names_in_

In [ ]:
input_features = entire_dataset[input_features_columns]

In [ ]:
predictions = trained_xgb_model.predict(input_features)

In [ ]:
synthetic_dataset = entire_dataset.drop(columns=input_features_columns)

In [ ]:
synthetic_dataset["predictions"] = predictions

In [ ]:
synthetic_dataset.head()

In [ ]:
synthetic_dataset = synthetic_dataset.drop(
    columns=[
        "local_year",
        "load_mw",
        "year_electricity_demand_mw",
    ]
)

In [ ]:
synthetic_dataset.head()

In [ ]:
synthetic_dataset.to_parquet(
    "./data/synthetic_dataset.parquet", engine="pyarrow"
)